# Electron g-2 Winner – 10 Billion Paths × 30 Repeats

**Configuration**: walk_length=28, phase_scale=39.0, fbs_power=7.5 (weighted winner)
**Weighting**: amp *= 1.0 / (step + 1) ** 1.8
**Paths per run**: 10 billion
**Repeats**: 30 (overnight run for tight final statistics)

Results saved to: winner_10B_30repeats_results.csv
Beep sound plays when finished.

In [ ]:
import cupy as cp
import numpy as np
from tqdm import tqdm
import time
import pandas as pd
import winsound

# ====================== SETTINGS ======================
N_paths_total = 10_000_000_000   # 10 billion – overnight run
BATCH_SIZE    = 15_000_000       # conservative for 10B paths on RTX 3060 Ti
N_REPEATS     = 30               # 30 repeats for good statistics

C = 9.6e-7
alpha = 1 / 137.035999084

configs = [
    {'name': 'Winner: 28/39/7.5 weighted 1.8', 'walk_length': 28, 'phase_scale': 39.0, 'fbs_power': 7.5},
]

n_batches = (N_paths_total + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Total paths per run: {N_paths_total:,}")
print(f"Repeats: {N_REPEATS} | Total runs: {len(configs) * N_REPEATS}")
print(f"Estimated time: ~{N_REPEATS * 90 / 3600:.1f}–{N_REPEATS * 120 / 3600:.1f} hours (90–120 s per run)")

In [ ]:
# ====================== LOAD FIXED ASSETS ======================
phi = (1 + np.sqrt(5)) / 2
inv_phi = 1 / phi

def generate_600cell_vertices():
    coords = []
    for signs in np.array(np.meshgrid(*[[-0.5,0.5]]*4)).T.reshape(-1,4):
        coords.append(signs)
    for i in range(4):
        for s in [-1,1]:
            v = np.zeros(4); v[i] = s; coords.append(v)
    base = [0.5*phi, 0.5, 0.5*inv_phi, 0]
    for perm in [[0,1,2,3],[0,1,3,2],[0,2,1,3],[0,2,3,1],[0,3,1,2],[0,3,2,1]]:
        for signs in [[1,1,1],[1,1,-1],[1,-1,1],[-1,1,1]]:
            v = np.zeros(4)
            v[perm[0]] = signs[0] * base[0]
            v[perm[1]] = signs[1] * base[1]
            v[perm[2]] = signs[2] * base[2]
            coords.append(v)
    verts = np.unique(np.round(coords, decimals=10), axis=0)
    verts = verts / np.linalg.norm(verts, axis=1)[:, None]
    return cp.asarray(verts, dtype=cp.float32)

vertices = generate_600cell_vertices()
N_VERT = vertices.shape[0]

dists = cp.linalg.norm(vertices[:, None] - vertices[None, :], axis=-1)
edge_threshold = cp.sort(dists.flatten())[720*2 + 1]
adj_mask = (dists < edge_threshold + 1e-6) & (dists > 1e-6)
neighbors = [cp.argwhere(adj_mask[i])[:,0] for i in range(N_VERT)]

print(f"Loaded: {N_VERT} vertices, avg degree ~{np.mean([len(n) for n in neighbors]):.1f}")

In [ ]:
# ====================== RUN WINNER 30 TIMES ======================
all_results = []
global_start = time.time()

cfg = configs[0]  # only one config
print(f"\n=== Running winner: {cfg['name']} ===")

for rep in range(1, N_REPEATS + 1):
    print(f"\nRepeat {rep}/{N_REPEATS}")
    start = time.time()

    raw_amplitudes = cp.zeros(3, dtype=cp.complex64)

    for batch_idx in tqdm(range(n_batches), desc="Batches", leave=False):
        batch_size = min(BATCH_SIZE, N_paths_total - batch_idx * BATCH_SIZE)
        current = cp.random.randint(0, N_VERT, batch_size)
        phase = cp.zeros(batch_size, dtype=cp.complex64)
        fbs_grade = cp.ones(batch_size, dtype=cp.float32)

        for step in range(cfg['walk_length']):
            neigh_choices = cp.array([cp.random.choice(n, size=1)[0] for n in neighbors])[current]
            edge_len = cp.linalg.norm(vertices[current] - vertices[neigh_choices], axis=1)
            phase += 1j * cfg['phase_scale'] * edge_len * fbs_grade
            fbs_grade *= 1.0 / (edge_len ** cfg['fbs_power'] + 0.1)
            current = neigh_choices

        amp = cp.exp(phase) * fbs_grade

        # ──────────────── PATH-LENGTH WEIGHTING ────────────────
        weight = 1.0 / (step + 1) ** 1.8
        amp *= weight
        # ──────────────────────────────────────────────────────

        e_amp = amp * cp.exp(1j * 0.0)
        h_amp = amp * cp.exp(1j * np.pi / 2)
        q_amp = amp * cp.exp(1j * np.pi)

        raw_amplitudes[0] += cp.sum(e_amp)
        raw_amplitudes[1] += cp.sum(h_amp)
        raw_amplitudes[2] += cp.sum(q_amp)

        cp.cuda.Device().synchronize()  # flush GPU

    interference = raw_amplitudes / N_paths_total
    abs_interf = cp.abs(interference)
    total_mag = cp.sum(abs_interf)
    mean_amp = float(cp.mean(cp.abs(amp)))

    S = alpha / (2 * np.pi) * mean_amp
    mixing_sum = float(abs_interf[2] / total_mag) + 0.7 * float(abs_interf[1] / total_mag)
    delta_mu_e = C * mixing_sum * S

    result = {
        'config': cfg['name'],
        'repeat': rep,
        'delta_mu_e': delta_mu_e,
        'mean_amp': mean_amp,
        'S': S,
        'raw_sum_mag': float(cp.abs(raw_amplitudes).sum()),
        'time_sec': time.time() - start
    }
    all_results.append(result)

    print(f"  δμ_e: {delta_mu_e:.3e} | mean_amp: {mean_amp:.3e} | time: {result['time_sec']:.1f}s")

# Final summary
df = pd.DataFrame(all_results)
df.to_csv("winner_10B_30repeats_results.csv", index=False)

print("\n=== FINAL SUMMARY ===")
print(f"Mean δμ_e:    {df['delta_mu_e'].mean():.3e}")
print(f"Median δμ_e:  {df['delta_mu_e'].median():.3e}")
print(f"Std δμ_e:     {df['delta_mu_e'].std():.3e}")
print(f"Best δμ_e:    {df['delta_mu_e'].min():.3e}")
print(f"Worst δμ_e:   {df['delta_mu_e'].max():.3e}")
print(f"95% CI low:   {df['delta_mu_e'].quantile(0.025):.3e}")
print(f"95% CI high:  {df['delta_mu_e'].quantile(0.975):.3e}")
print(f"Total time:   {(time.time() - global_start)/3600:.2f} hours")

print("\nResults saved to winner_10B_30repeats_results.csv")

# Beep when done (Windows)
winsound.Beep(1200, 2000)  # 1200 Hz, 2 seconds
print("FINISHED! You can check the results now.")

## Sorted Results Table
Run this after completion:

In [ ]:
df = pd.read_csv("winner_10B_30repeats_results.csv")
print("Sorted by δμ_e (lowest first):")
print(df.sort_values('delta_mu_e').to_string(index=False))